# Steam 리뷰 감성 + 이슈 유형화 분석

In [1]:
import os
import json
import time
import asyncio
import platform
from pathlib import Path
from typing import List, Literal, Optional

import pandas as pd
import numpy as np
from IPython.display import display
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModelSettings
from tqdm.auto import tqdm

c:\Users\joon5\Documents\github\steam-indie-game-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


uv pip install "pydantic-ai-slim[google]" tqdm pandas python-dotenv

In [2]:
import numpy as np
import pandas as pd
import math
from datetime import datetime

def to_serializable(obj):
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    elif isinstance(obj, tuple):
        return [to_serializable(v) for v in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        # nan 방지
        if np.isnan(obj):
            return None
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    elif isinstance(obj, datetime):
        return obj.isoformat()
    elif pd.isna(obj):
        return None
    else:
        return obj
    
# ai가 내주는 솔루션, 왜 작동 가능한지 나도 몰?루

# 환경설정

In [3]:
from pathlib import Path
import pandas as pd

# 프로젝트 루트 직접 지정
ROOT = Path(r"C:\Users\joon5\Documents\github\steam-indie-game-analysis")

# data/processed 폴더
DATA_DIR = ROOT / "data" / "processed"

# 파일 경로
SAMPLE_PATH = DATA_DIR / "steam_stratified_sample_v4.csv"
HIST_PATH   = DATA_DIR / "review_histogram_v4.csv"
FULL_PATH   = DATA_DIR / "steam_indie_list_202604211615.csv"

In [4]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
gemini_model = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
model_id = f"google-gla:{gemini_model}"

print("API 키 설정 확인:", "O" if api_key else "X")
print("사용 모델:", model_id)

# 파일 경로
# 입력 파일
INPUT_PATH = DATA_DIR / "steam_indie_reviews_202604230927.csv"

# 결과 저장 폴더
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "steam_review_llm_checkpoint.json"
RESULT_JSON_PATH = OUTPUT_DIR / "steam_review_llm_results.json"
RESULT_CSV_PATH = OUTPUT_DIR / "steam_review_llm_results.csv"
TAG_CSV_PATH = OUTPUT_DIR / "steam_review_llm_issue_tags.csv"
SUMMARY_CSV_PATH = OUTPUT_DIR / "steam_review_llm_summary.csv"

# 실행 옵션
# REVIEWS_PER_GAME = 5             # 처음 테스트는 50~100 추천 / 전체 돌릴 땐 None
# BATCH_SIZE = 4
# MAX_CONCURRENT = 2
# MAX_RETRIES = 4
# MIN_REVIEW_LEN = 15
# MAX_REVIEW_CHARS = 2500   # 너무 긴 리뷰는 비용/실패 방지를 위해 잘라서 사용

REVIEWS_PER_GAME = 5
BATCH_SIZE = 1
MAX_CONCURRENT = 1
MAX_RETRIES = 2
MAX_REVIEW_CHARS = 1500

# 필터 옵션
ONLY_ENGLISH = True
ONLY_STEAM_PURCHASE = True
EXCLUDE_FREE_RECEIVED = True

# 비용 대략 추정용 (원하면 수정)
INPUT_PRICE_PER_1M = 0.25
OUTPUT_PRICE_PER_1M = 0.50
USD_TO_KRW = 1500

API 키 설정 확인: O
사용 모델: google-gla:gemini-2.5-flash


실행 옵션 설정
```
테스트는
TEST_N = 100

그 다음
TEST_N = None
```

```
실 사용 때
TEST_N = 50
BATCH_SIZE = 3
MAX_CONCURRENT = 1

그 다음
TEST_N = 200
BATCH_SIZE = 5
MAX_CONCURRENT = 2

마지막에
TEST_N = None
```

In [5]:
# 무료 티어 대응용 실행 옵션
# - 호출 수를 줄이기 위해 게임당 리뷰 수 축소
# - 한 번에 1개 리뷰만 보내서 배치 실패 부담 축소
# - 동시 호출 제거
# - 재시도 횟수 축소 (무료 티어에서는 재시도도 쿼터를 빨리 소모함)
# - 긴 리뷰는 일부만 잘라서 전송
REVIEWS_PER_GAME = 5
BATCH_SIZE = 1
MAX_CONCURRENT = 1
MAX_RETRIES = 2
MIN_REVIEW_LEN = 15
MAX_REVIEW_CHARS = 1500

# 무료 티어 rate limit 대응
FREE_TIER_SLEEP_SEC = 15
CHUNK_SIZE = 1

# 필터 옵션
ONLY_ENGLISH = True
ONLY_STEAM_PURCHASE = True
EXCLUDE_FREE_RECEIVED = True

# 비용 대략 추정용
INPUT_PRICE_PER_1M = 0.25
OUTPUT_PRICE_PER_1M = 0.50
USD_TO_KRW = 1500

# 데이터 로드

In [6]:
selected_games = {
    1299690: "Gori: Cuddly Carnage",
    1948800: "Yi Xian: The Cultivation Card Game",
    1272320: "Diplomacy is Not an Option",
}

# ai가 랜덤하게 선정한거라 무슨 게임인지 모름

# 데이터 전처리

In [7]:
df = pd.read_csv(INPUT_PATH)

print("원본 데이터 크기:", df.shape)
print("원본 appid 개수:", df["appid"].nunique())

required_cols = [
    "recommendationid",
    "appid",
    "language",
    "review",
    "timestamp_created",
    "voted_up",
    "steam_purchase",
    "received_for_free",
    "author_playtime_at_review",
]

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"필수 컬럼이 없습니다: {missing_cols}")

원본 데이터 크기: (13106, 21)
원본 appid 개수: 73


In [8]:
# review 텍스트 정리
df["review"] = df["review"].fillna("").astype(str).str.strip()

# 언어 영어만
df["language"] = df["language"].fillna("").astype(str).str.strip().str.lower()

# 최소 길이
df = df[df["review"].str.len() >= MIN_REVIEW_LEN].copy()

df = df[df["language"] == "english"].copy()

# 언어 필터
if ONLY_ENGLISH:
    df = df[df["language"].fillna("").str.lower() == "english"].copy()

# 실구매 필터
if ONLY_STEAM_PURCHASE:
    df = df[df["steam_purchase"] == True].copy()

# 무료 수령 제외
if EXCLUDE_FREE_RECEIVED:
    df = df[df["received_for_free"] == False].copy()

# 중복 제거
df = df.drop_duplicates(subset=["recommendationid"]).reset_index(drop=True)

df = df[df["appid"].isin(selected_games.keys())].copy()
df["game_name"] = df["appid"].map(selected_games)

print("\n선택 게임 필터 후 데이터 크기:", df.shape)
print(df.groupby(["appid", "game_name"]).size())


선택 게임 필터 후 데이터 크기: (140, 22)
appid    game_name                         
1272320  Diplomacy is Not an Option            44
1299690  Gori: Cuddly Carnage                  71
1948800  Yi Xian: The Cultivation Card Game    25
dtype: int64


# 게임당 셈플링

In [9]:
sampled_list = []

for appid, g in df.groupby("appid"):
    n = min(REVIEWS_PER_GAME, len(g))
    sampled = g.sample(n=n, random_state=42)
    sampled_list.append(sampled)

df = pd.concat(sampled_list, ignore_index=True)
df = df.sort_values(["appid", "timestamp_created"]).reset_index(drop=True)

df["review_for_llm"] = df["review"].str.slice(0, MAX_REVIEW_CHARS)

print("\n샘플링 후 데이터 크기:", df.shape)
print(df.groupby(["appid", "game_name"]).size())



샘플링 후 데이터 크기: (15, 23)
appid    game_name                         
1272320  Diplomacy is Not an Option            5
1299690  Gori: Cuddly Carnage                  5
1948800  Yi Xian: The Cultivation Card Game    5
dtype: int64


# 핼퍼 함수

In [10]:
def get_playtime_stage(minutes):
    if pd.isna(minutes):
        return "unknown"
    if minutes < 30:
        return "very_early"
    elif minutes < 120:
        return "early"
    elif minutes < 600:
        return "mid"
    else:
        return "late"


def load_checkpoint(path=CHECKPOINT_PATH):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_checkpoint(results, path=CHECKPOINT_PATH):
    safe_results = to_serializable(results)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_results, f, ensure_ascii=False, indent=2)


def print_cost_report(input_tokens, output_tokens):
    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_1M
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
    total_cost_usd = input_cost + output_cost
    total_cost_krw = total_cost_usd * USD_TO_KRW

    print("\n" + "=" * 60)
    print("토큰 사용량 / 예상 비용")
    print("=" * 60)
    print(f"입력 토큰: {input_tokens:,}")
    print(f"출력 토큰: {output_tokens:,}")
    print(f"예상 비용(USD): ${total_cost_usd:.6f}")
    print(f"예상 비용(KRW): ₩{total_cost_krw:,.2f}")
    print("=" * 60)

# 4. Pydantic 출력 스키마

문제: 영어가 아닌 다른 언어일 경우엔 따로 설정해야 하나?????

In [11]:
class IssueTag(BaseModel):
    category: Literal[
        "bug",
        "optimization",
        "balance",
        "content_lack",
        "difficulty",
        "ui_ux",
        "translation",
        "multiplayer",
        "controls",
        "price_value",
        "story",
        "community",
        "other",
    ] = Field(description="리뷰에서 언급된 이슈 카테고리")

    sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="이 이슈에 대한 감정"
    )

    evidence: str = Field(
        description="원문을 바탕으로 한 짧은 근거",
        min_length=3,
        max_length=120
    )


class SteamReviewAnalysis(BaseModel):
    recommendationid: str = Field(description="입력 리뷰 ID 그대로 반환")

    llm_sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="리뷰 본문 기준 감정"
    )

    sentiment_score: int = Field(
        ge=1,
        le=5,
        description="1=매우 부정, 3=중립, 5=매우 긍정"
    )

    primary_issue: Literal[
        "bug",
        "optimization",
        "balance",
        "content_lack",
        "difficulty",
        "ui_ux",
        "translation",
        "multiplayer",
        "controls",
        "price_value",
        "story",
        "community",
        "praise",
        "other",
    ] = Field(description="가장 핵심적인 이슈 1개")

    urgency: Literal["low", "medium", "high"] = Field(
        description="개선 우선순위용 긴급도"
    )

    playtime_stage: Literal["very_early", "early", "mid", "late", "unknown"] = Field(
        description="리뷰 작성 시점 플레이타임 단계"
    )

    issue_tags: List[IssueTag] = Field(
        description="실제로 언급된 이슈만 포함",
        max_length=5
    )

    one_line_summary: str = Field(
        description="리뷰 핵심 한 줄 요약",
        min_length=5,
        max_length=200
    )

    suggested_action: str = Field(
        description="개발사 입장에서 참고할 액션 1문장",
        min_length=5,
        max_length=200
    )


class BatchSteamReviewAnalysis(BaseModel):
    results: List[SteamReviewAnalysis]

# Agent 생성

In [12]:
system_prompt = """
당신은 Steam 게임 리뷰 분석 전문가입니다.

각 리뷰에 대해 다음을 판단하세요.
1. 리뷰 본문 기준 감정 (llm_sentiment)
2. 가장 핵심적인 이슈(primary_issue)
3. 세부 이슈(issue_tags)
4. 개발사 관점 suggested_action

중요 규칙:
- recommendationid는 반드시 입력값 그대로 반환하세요.
- voted_up은 참고 정보일 뿐, 감정은 review 텍스트 기준으로 판단하세요.
- 추천 리뷰라도 불만이 많으면 mixed 또는 negative로 판단할 수 있습니다.
- 비추천 리뷰라도 장단점이 섞여 있으면 mixed로 판단할 수 있습니다.
- issue_tags에는 실제로 언급된 것만 넣으세요.
- 근거(evidence)는 짧고 명확하게 작성하세요.
- review가 매우 짧거나 밈/농담 위주면 과잉 해석하지 마세요.
"""

review_agent = Agent(
    model_id,
    output_type=BatchSteamReviewAnalysis,
    system_prompt=system_prompt,
)

review_settings = GoogleModelSettings(
    temperature=0.2
)

# 프롬프트

In [13]:
def build_batch_prompt(batch_df):
    blocks = []

    for _, row in batch_df.iterrows():
        rid = str(row["recommendationid"])
        appid = row["appid"]
        steam_label = "positive" if bool(row["voted_up"]) else "negative"
        playtime = row["author_playtime_at_review"]
        playtime_stage = get_playtime_stage(playtime)
        text = row["review_for_llm"]

        block = f"""
---
[recommendationid: {rid}]
[appid: {appid}]
[steam_label: {steam_label}]
[author_playtime_at_review_minutes: {playtime}]
[playtime_stage_hint: {playtime_stage}]
[review_text]
{text}
"""
        blocks.append(block)

    prompt = (
        f"다음 {len(batch_df)}개의 Steam 리뷰를 각각 분석해주세요.\n"
        "반드시 입력된 recommendationid를 그대로 유지해서 반환하세요.\n\n"
        + "\n".join(blocks)
    )
    return prompt

# 비동기 분석

In [14]:
sem = asyncio.Semaphore(MAX_CONCURRENT)


async def analyze_batch(batch_df, all_results, stats, pbar):
    async with sem:
        prompt = build_batch_prompt(batch_df)

        for attempt in range(MAX_RETRIES):
            try:
                # result = await review_agent.run(
                #     prompt,
                #     model_settings=review_settings
                # )
                result = await review_agent.run(prompt)

                output = result.output

                # usage 집계
                try:
                    usage = result.usage()
                    stats["input_tokens"] += getattr(usage, "input_tokens", 0) or 0
                    stats["output_tokens"] += getattr(usage, "output_tokens", 0) or 0
                except Exception:
                    pass

                stats["requests"] += 1

                input_ids = set(batch_df["recommendationid"].astype(str).tolist())
                matched_ids = set()

                for item in output.results:
                    rid = str(item.recommendationid)

                    if rid not in input_ids:
                        continue

                    row = batch_df[batch_df["recommendationid"].astype(str) == rid].iloc[0]
                    matched_ids.add(rid)

                    steam_label_text = "positive" if bool(row["voted_up"]) else "negative"

                    record = {
                        "recommendationid": rid,
                        "appid": row["appid"],
                        "language": row["language"],
                        "steam_voted_up": bool(row["voted_up"]),
                        "steam_label_text": steam_label_text,
                        "timestamp_created": row["timestamp_created"],
                        "author_playtime_at_review": row["author_playtime_at_review"],
                        "author_playtime_forever": row.get("author_playtime_forever", None),
                        "votes_up": row.get("votes_up", None),
                        "votes_funny": row.get("votes_funny", None),
                        "weighted_vote_score": row.get("weighted_vote_score", None),
                        "comment_count": row.get("comment_count", None),
                        "llm_sentiment": item.llm_sentiment,
                        "sentiment_score": item.sentiment_score,
                        "primary_issue": item.primary_issue,
                        "urgency": item.urgency,
                        "playtime_stage": item.playtime_stage,
                        "one_line_summary": item.one_line_summary,
                        "suggested_action": item.suggested_action,
                        "issue_tags": [x.model_dump() for x in item.issue_tags],
                        "review": row["review"],
                        "sentiment_match": (
                            "match"
                            if (
                                (steam_label_text == "positive" and item.llm_sentiment in ["positive", "mixed"])
                                or
                                (steam_label_text == "negative" and item.llm_sentiment in ["negative", "mixed"])
                            )
                            else "mismatch"
                        )
                    }

                    all_results.append(record)

                # 혹시 누락된 리뷰가 있으면 에러 기록
                missing_ids = input_ids - matched_ids
                for missing_id in missing_ids:
                    row = batch_df[batch_df["recommendationid"].astype(str) == missing_id].iloc[0]

                    all_results.append({
                        "recommendationid": str(missing_id),
                        "appid": row["appid"],
                        "language": row["language"],
                        "steam_voted_up": bool(row["voted_up"]),
                        "steam_label_text": "positive" if bool(row["voted_up"]) else "negative",
                        "timestamp_created": row["timestamp_created"],
                        "author_playtime_at_review": row["author_playtime_at_review"],
                        "author_playtime_forever": row.get("author_playtime_forever", None),
                        "votes_up": row.get("votes_up", None),
                        "votes_funny": row.get("votes_funny", None),
                        "weighted_vote_score": row.get("weighted_vote_score", None),
                        "comment_count": row.get("comment_count", None),
                        "llm_sentiment": None,
                        "sentiment_score": None,
                        "primary_issue": None,
                        "urgency": None,
                        "playtime_stage": None,
                        "one_line_summary": None,
                        "suggested_action": None,
                        "issue_tags": [],
                        "review": row["review"],
                        "sentiment_match": "missing_output"
                    })

                pbar.update(len(batch_df))
                return

            except Exception as e:
                wait_sec = min(3 * (2 ** attempt), 60)
                print(f"[재시도 {attempt + 1}/{MAX_RETRIES}] 오류:", e)

                if attempt < MAX_RETRIES - 1:
                    await asyncio.sleep(wait_sec)
                else:
                    print("[최종 실패] 이 배치는 실패 처리")
                    for _, row in batch_df.iterrows():
                        all_results.append({
                            "recommendationid": str(row["recommendationid"]),
                            "appid": row["appid"],
                            "language": row["language"],
                            "steam_voted_up": bool(row["voted_up"]),
                            "steam_label_text": "positive" if bool(row["voted_up"]) else "negative",
                            "timestamp_created": row["timestamp_created"],
                            "author_playtime_at_review": row["author_playtime_at_review"],
                            "author_playtime_forever": row.get("author_playtime_forever", None),
                            "votes_up": row.get("votes_up", None),
                            "votes_funny": row.get("votes_funny", None),
                            "weighted_vote_score": row.get("weighted_vote_score", None),
                            "comment_count": row.get("comment_count", None),
                            "llm_sentiment": None,
                            "sentiment_score": None,
                            "primary_issue": None,
                            "urgency": None,
                            "playtime_stage": None,
                            "one_line_summary": None,
                            "suggested_action": None,
                            "issue_tags": [],
                            "review": row["review"],
                            "sentiment_match": "batch_failed"
                        })
                    pbar.update(len(batch_df))
                    return


async def run_analysis(df_input):
    checkpoint_data = load_checkpoint()
    done_ids = {str(x["recommendationid"]) for x in checkpoint_data}

    df_work = df_input[~df_input["recommendationid"].astype(str).isin(done_ids)].copy()
    df_work = df_work.reset_index(drop=True)

    print("이미 처리된 리뷰 수:", len(done_ids))
    print("이번에 처리할 리뷰 수:", len(df_work))

    all_results = checkpoint_data.copy()
    stats = {
        "input_tokens": 0,
        "output_tokens": 0,
        "requests": 0,
    }

    tasks = []
    pbar = tqdm(total=len(df_work), desc="LLM 리뷰 분석")

    for start in range(0, len(df_work), BATCH_SIZE):
        batch_df = df_work.iloc[start:start + BATCH_SIZE]
        tasks.append(analyze_batch(batch_df, all_results, stats, pbar))

    # 너무 한꺼번에 몰지 않도록 나눠서 실행
    # chunk_size = MAX_CONCURRENT * 5
    chunk_size = 1

    # for i in range(0, len(tasks), chunk_size):
    #     chunk = tasks[i:i + chunk_size]
    #     await asyncio.gather(*chunk)
    #     save_checkpoint(all_results)

    for i in range(0, len(tasks), chunk_size):
        chunk = tasks[i:i + chunk_size]
        await asyncio.gather(*chunk)
        save_checkpoint(all_results)

        # 무료 티어 분당 제한 대응
        await asyncio.sleep(15)

    pbar.close()

    return all_results, stats

# 실행

In [15]:
results, stats = await run_analysis(df)

print_cost_report(
    input_tokens=stats["input_tokens"],
    output_tokens=stats["output_tokens"]
)


# 결과 저장
safe_results = to_serializable(results)

with open(RESULT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(safe_results, f, ensure_ascii=False, indent=2)

df_result = pd.DataFrame(safe_results)
df_result.to_csv(RESULT_CSV_PATH, index=False, encoding="utf-8-sig")

print("JSON 저장:", RESULT_JSON_PATH)
print("CSV 저장:", RESULT_CSV_PATH)

df_result.head()

이미 처리된 리뷰 수: 5
이번에 처리할 리뷰 수: 10


LLM 리뷰 분석:   0%|          | 0/10 [00:00<?, ?it/s]

[재시도 1/2] 오류: Exceeded maximum retries (1) for output validation


LLM 리뷰 분석: 100%|██████████| 10/10 [03:52<00:00, 23.22s/it]


토큰 사용량 / 예상 비용
입력 토큰: 14,571
출력 토큰: 10,343
예상 비용(USD): $0.008814
예상 비용(KRW): ₩13.22
JSON 저장: outputs\steam_review_llm_results.json
CSV 저장: outputs\steam_review_llm_results.csv


,recommendationid,appid,language,steam_voted_up,steam_label_text,timestamp_created,author_playtime_at_review,author_playtime_forever,votes_up,votes_funny,...,llm_sentiment,sentiment_score,primary_issue,urgency,playtime_stage,one_line_summary,suggested_action,issue_tags,review,sentiment_match
0,157971451,1948800,english,True,positive,1707539729,2404,3794,9,0,...,positive,5,praise,low,late,"탁월한 무료 로그라이크 게임으로, 훌륭한 실시간 전략 플레이와 지속적인 업데이트가 ...",영문 번역의 완성도를 높이고 업데이트 시 발생하는 서버 문제를 개선하여 플레이어 경...,"[{'category': 'translation', 'sentiment': 'neg...",Phenomenal game. It is unbelievable that you c...,match
1,158566335,1948800,english,True,positive,1708206157,4558,11349,2,0,...,mixed,3,balance,high,late,"다양한 시스템과 창의적인 오토 체스 카드 게임 메커니즘은 긍정적이나, 선두 플레이어...","뒤쳐진 플레이어가 따라잡을 수 있는 메커니즘을 추가하여 게임 내 균형을 개선하고, ...","[{'category': 'balance', 'sentiment': 'negativ...",Good:\n\nSystem experience diversity\nSeveral ...,match
2,159919395,1948800,english,True,positive,1709662308,5353,7535,0,0,...,positive,5,praise,low,late,"뛰어난 제작 퀄리티와 깊이 있는 시스템을 갖춘 무료 덱빌딩 오토배틀러로, 현지화와 ...",게임의 높은 완성도와 무료 플레이 모델의 가치를 지속적으로 홍보하여 사용자 유입을 ...,"[{'category': 'other', 'sentiment': 'positive'...",A genius blend of deckbuilder and autobattler ...,match
3,161657039,1948800,english,True,positive,1711638698,1891,3309,0,0,...,positive,5,praise,low,late,"페이투윈 요소 없고, 과금 유도 없으며, 서버가 안정적이라는 긍정적인 리뷰입니다.",현재의 합리적인 과금 모델과 안정적인 서버 운영을 유지하세요.,"[{'category': 'price_value', 'sentiment': 'pos...",Wuxia cultivator card duel.\nNot pay to win. I...,match
4,162644942,1948800,english,True,positive,1712812226,6684,19314,0,0,...,positive,5,praise,low,late,"장시간 플레이해도 질리지 않는 잘 만들어진 게임으로, 훌륭한 밸런스와 공정한 가격 ...","영문 번역 품질을 개선하여 신규 플레이어의 접근성을 높이고, 현재의 뛰어난 게임 밸...","[{'category': 'translation', 'sentiment': 'neg...",Saw the fun playtime and had to recommend.\nTh...,match


In [16]:
print(df_result.columns.tolist())

['recommendationid', 'appid', 'language', 'steam_voted_up', 'steam_label_text', 'timestamp_created', 'author_playtime_at_review', 'author_playtime_forever', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'llm_sentiment', 'sentiment_score', 'primary_issue', 'urgency', 'playtime_stage', 'one_line_summary', 'suggested_action', 'issue_tags', 'review', 'sentiment_match']


# issue_tags 펼치기

In [17]:
# game_name 복구
if "game_name" not in df_result.columns:
    df_result["game_name"] = df_result["appid"].map(selected_games)

flat_rows = []

for _, row in df_result.iterrows():
    tags = row["issue_tags"]

    if isinstance(tags, str):
        try:
            tags = json.loads(tags)
        except Exception:
            tags = []

    if not isinstance(tags, list):
        tags = []

    for tag in tags:
        flat_rows.append({
            "recommendationid": row["recommendationid"],
            "appid": row["appid"],
            "game_name": row["game_name"],
            "steam_label_text": row["steam_label_text"],
            "llm_sentiment": row["llm_sentiment"],
            "primary_issue": row["primary_issue"],
            "tag_category": tag.get("category"),
            "tag_sentiment": tag.get("sentiment"),
            "tag_evidence": tag.get("evidence"),
        })

df_tags = pd.DataFrame(flat_rows)
df_tags.to_csv(TAG_CSV_PATH, index=False, encoding="utf-8-sig")

print("세부 이슈 태그 CSV 저장:", TAG_CSV_PATH)
df_tags.head()


세부 이슈 태그 CSV 저장: outputs\steam_review_llm_issue_tags.csv


,recommendationid,appid,game_name,steam_label_text,llm_sentiment,primary_issue,tag_category,tag_sentiment,tag_evidence
0,157971451,1948800,Yi Xian: The Cultivation Card Game,positive,positive,praise,translation,negative,The story is a bit rough around the edges in t...
1,157971451,1948800,Yi Xian: The Cultivation Card Game,positive,positive,praise,multiplayer,negative,server problems when they do updates
2,158566335,1948800,Yi Xian: The Cultivation Card Game,positive,mixed,balance,balance,negative,Lack of mechanics to close gap between leading...
3,158566335,1948800,Yi Xian: The Cultivation Card Game,positive,mixed,balance,other,negative,"Card combos require specific sequence, making ..."
4,159919395,1948800,Yi Xian: The Cultivation Card Game,positive,positive,praise,other,positive,"top-notch production. Mechanics, music, sound,..."


# 요약 통계

In [ ]:
summary_rows = []

if len(df_result) > 0:
    sentiment_summary = (
        df_result.groupby(["game_name", "llm_sentiment"])
        .size()
        .reset_index(name="count")
    )
    sentiment_summary["summary_type"] = "llm_sentiment"
    summary_rows.append(sentiment_summary)

    issue_summary = (
        df_result.groupby(["game_name", "primary_issue"])
        .size()
        .reset_index(name="count")
    )
    issue_summary["summary_type"] = "primary_issue"
    summary_rows.append(issue_summary)

    match_summary = (
        df_result.groupby(["game_name", "sentiment_match"])
        .size()
        .reset_index(name="count")
    )
    match_summary["summary_type"] = "sentiment_match"
    summary_rows.append(match_summary)

if len(df_tags) > 0:
    tag_summary = (
        df_tags.groupby(["game_name", "tag_category"])
        .size()
        .reset_index(name="count")
    )
    tag_summary["summary_type"] = "issue_tag"
    summary_rows.append(tag_summary)

if summary_rows:
    df_summary = pd.concat(summary_rows, ignore_index=True)
    df_summary.to_csv(SUMMARY_CSV_PATH, index=False, encoding="utf-8-sig")
    display(df_summary.head())

display("=== 게임별 감정 분포 ===")
display(pd.crosstab(df_result["game_name"], df_result["llm_sentiment"]))

display("=== 게임별 핵심 이슈 분포 ===")
display(pd.crosstab(df_result["game_name"], df_result["primary_issue"]))

display("=== 게임별 Steam 라벨 vs LLM 일치 여부 ===")
display(pd.crosstab(df_result["game_name"], df_result["sentiment_match"]))

if len(df_tags) > 0:
    display("=== 게임별 세부 이슈 태그 분포 ===")
    display(pd.crosstab(df_tags["game_name"], df_tags["tag_category"]))

,game_name,llm_sentiment,count,summary_type,primary_issue,sentiment_match,tag_category
0,Diplomacy is Not an Option,mixed,1,llm_sentiment,NaN,NaN,NaN
1,Diplomacy is Not an Option,negative,2,llm_sentiment,NaN,NaN,NaN
2,Diplomacy is Not an Option,positive,2,llm_sentiment,NaN,NaN,NaN
3,Gori: Cuddly Carnage,positive,5,llm_sentiment,NaN,NaN,NaN
4,Yi Xian: The Cultivation Card Game,mixed,1,llm_sentiment,NaN,NaN,NaN


'=== 게임별 감정 분포 ==='

llm_sentiment,mixed,negative,positive
game_name,,,
Diplomacy is Not an Option,1,2,2
Gori: Cuddly Carnage,0,0,5
Yi Xian: The Cultivation Card Game,1,0,4


'=== 게임별 핵심 이슈 분포 ==='

primary_issue,balance,difficulty,praise
game_name,,,
Diplomacy is Not an Option,1,2,2
Gori: Cuddly Carnage,0,0,5
Yi Xian: The Cultivation Card Game,1,0,4


'=== 게임별 Steam 라벨 vs LLM 일치 여부 ==='

sentiment_match,match
game_name,
Diplomacy is Not an Option,5
Gori: Cuddly Carnage,5
Yi Xian: The Cultivation Card Game,5


'=== 게임별 세부 이슈 태그 분포 ==='

tag_category,balance,content_lack,controls,difficulty,multiplayer,other,price_value,story,translation
game_name,,,,,,,,,
Diplomacy is Not an Option,2,0,1,4,0,2,0,1,0
Gori: Cuddly Carnage,0,0,1,1,0,2,0,2,0
Yi Xian: The Cultivation Card Game,2,1,0,2,2,2,3,0,3


: 